In [2]:
import argparse
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# --------------------------------------------------------------------
# Reprodutibilidade
# --------------------------------------------------------------------
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------
# Modelos: MLPs totalmente conectados com ativação ReLU
# --------------------------------------------------------------------
class MLP(nn.Module):
    """MLP simples: 784 -> hidden -> hidden -> 10, com dropout opcional."""

    def __init__(self, hidden_size: int, dropout: float = 0.0):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 10)  # logits (sem softmax)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        x = self.drop(x)
        logits = self.fc3(x)  # retorna logits; softmax é aplicado fora
        return logits


# --------------------------------------------------------------------
# Dados: MNIST, com jitter opcional (translação de até 2 pixels),
# como mencionado no artigo para o professor.
# --------------------------------------------------------------------
def get_dataloaders(batch_size=128, jitter=False, data_dir="./data"):
    base_tf = [transforms.ToTensor()]

    if jitter:
        # "the input images were jittered by up to two pixels
        #  in any direction"
        train_tf = transforms.Compose(
            [transforms.RandomAffine(degrees=0, translate=(2 / 28, 2 / 28))]
            + base_tf
        )
    else:
        train_tf = transforms.Compose(base_tf)

    test_tf = transforms.Compose(base_tf)

    train_set = datasets.MNIST(
        data_dir, train=True, download=True, transform=train_tf
    )
    test_set = datasets.MNIST(
        data_dir, train=False, download=True, transform=test_tf
    )

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, num_workers=2
    )
    test_loader = DataLoader(
        test_set, batch_size=1000, shuffle=False, num_workers=2
    )
    return train_loader, test_loader


# --------------------------------------------------------------------
# Treino padrão (hard targets) - usado para o professor e para o
# aluno baseline.
# --------------------------------------------------------------------
def train_hard_targets(model, train_loader, test_loader, epochs, lr, device):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

        errors, acc = evaluate(model, test_loader, device)
        print(
            f"  [hard]  epoch {epoch:2d}/{epochs}  "
            f"test_acc={acc*100:.2f}%  test_errors={errors}"
        )
    return model


# --------------------------------------------------------------------
# Treino por destilação: combina
#   (a) entropia cruzada (KD) entre os soft targets do professor
#       (softmax com temperatura T) e o softmax do aluno na mesma T
#   (b) entropia cruzada padrão do aluno com os rótulos verdadeiros
#       (softmax em T=1)
# multiplicando o termo (a) por T^2, como recomendado no artigo,
# para manter a escala do gradiente comparável ao termo (b).
# --------------------------------------------------------------------
def distillation_loss(student_logits, teacher_logits, targets, T, alpha):
    """
    student_logits, teacher_logits: [batch, 10]
    targets: rótulos verdadeiros (hard targets), [batch]
    T: temperatura de destilação
    alpha: peso do termo soft (0 < alpha < 1); (1-alpha) vai para o termo hard
    """
    # termo soft: KL(soft_teacher || soft_student), na temperatura T
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    log_soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_loss = F.kl_div(
        log_soft_student, soft_teacher, reduction="batchmean"
    ) * (T * T)

    # termo hard: cross-entropy padrão em T=1
    hard_loss = F.cross_entropy(student_logits, targets)

    return alpha * soft_loss + (1.0 - alpha) * hard_loss


def train_distillation(
    student, teacher, train_loader, test_loader, epochs, lr, T, alpha, device
):
    student.to(device)
    teacher.to(device)
    teacher.eval()  # professor fica congelado, só gera soft targets

    optimizer = torch.optim.Adam(student.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        student.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            with torch.no_grad():
                teacher_logits = teacher(x)

            optimizer.zero_grad()
            student_logits = student(x)
            loss = distillation_loss(
                student_logits, teacher_logits, y, T=T, alpha=alpha
            )
            loss.backward()
            optimizer.step()

        errors, acc = evaluate(student, test_loader, device)
        print(
            f"  [dist T={T:>4}] epoch {epoch:2d}/{epochs}  "
            f"test_acc={acc*100:.2f}%  test_errors={errors}"
        )
    return student


# --------------------------------------------------------------------
# Avaliação
# --------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    acc = correct / total
    errors = total - correct
    return errors, acc


# --------------------------------------------------------------------
# Experimento especial do artigo: omitir todos os exemplos do dígito
# "3" do conjunto de transferência e verificar se o aluno destilado
# ainda consegue reconhecê-lo bem (Seção 3, parágrafo final).
# --------------------------------------------------------------------
def build_loader_without_digit(dataset, digit, batch_size=128):
    mask = dataset.targets != digit
    filtered = copy.deepcopy(dataset)
    filtered.data = filtered.data[mask]
    filtered.targets = filtered.targets[mask]
    return DataLoader(filtered, batch_size=batch_size, shuffle=True, num_workers=2)


def experiment_missing_digit(teacher, T, alpha, epochs, lr, device, data_dir):
    print("\n=== Experimento extra: aluno nunca vê o dígito '3' ===")
    train_tf = transforms.ToTensor()
    train_set = datasets.MNIST(
        data_dir, train=True, download=True, transform=train_tf
    )
    test_set = datasets.MNIST(
        data_dir, train=False, download=True, transform=train_tf
    )
    test_loader = DataLoader(test_set, batch_size=1000, shuffle=False)

    train_loader_no3 = build_loader_without_digit(train_set, digit=3)

    student = MLP(hidden_size=800, dropout=0.0)
    student = train_distillation(
        student, teacher, train_loader_no3, test_loader,
        epochs=epochs, lr=lr, T=T, alpha=alpha, device=device,
    )

    # Acurácia geral e específica na classe "3"
    student.eval()
    correct_all, total_all = 0, 0
    correct_3, total_3 = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = student(x).argmax(dim=1)
            correct_all += (pred == y).sum().item()
            total_all += y.size(0)
            mask3 = y == 3
            correct_3 += (pred[mask3] == y[mask3]).sum().item()
            total_3 += mask3.sum().item()

    print(
        f"  Acurácia geral: {correct_all/total_all*100:.2f}% "
        f"({total_all-correct_all} erros)"
    )
    print(
        f"  Acurácia no dígito '3' (nunca visto no treino): "
        f"{correct_3/total_3*100:.2f}% "
        f"({total_3-correct_3} erros em {total_3} exemplos)"
    )
    print(
        "  (No artigo original, o viés da classe '3' precisa ser "
        "ajustado manualmente para recuperar ~98.6% de acerto; aqui "
        "reportamos o resultado bruto, sem esse ajuste de viés.)"
    )


# --------------------------------------------------------------------
# Main
# --------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(
        description="Reprodução do experimento de destilação no MNIST (Hinton et al., 2015)"
    )
    parser.add_argument("--epochs-teacher", type=int, default=15)
    parser.add_argument("--epochs-student", type=int, default=15)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--teacher-hidden", type=int, default=1200)
    parser.add_argument("--student-hidden", type=int, default=800)
    parser.add_argument("--teacher-dropout", type=float, default=0.5)
    parser.add_argument(
        "--temperatures", type=float, nargs="+", default=[1, 2.5, 4, 8, 20]
    )
    parser.add_argument("--alpha", type=float, default=0.9,
                         help="Peso do termo soft (soft targets) na loss de destilação")
    parser.add_argument("--data-dir", type=str, default="./data")
    parser.add_argument("--run-missing-digit-experiment", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args, unknown = parser.parse_known_args() # Modified line

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Usando dispositivo: {device}\n")

    # ---- Dados -------------------------------------------------------
    train_loader_jitter, test_loader = get_dataloaders(
        batch_size=args.batch_size, jitter=True, data_dir=args.data_dir
    )
    train_loader_plain, _ = get_dataloaders(
        batch_size=args.batch_size, jitter=False, data_dir=args.data_dir
    )

    results = {}

    # ---- 1) Professor: rede grande, regularizada com dropout,
    #        treinada com jitter --------------------------------------
    print("=== Treinando o PROFESSOR (rede grande, com dropout) ===")
    teacher = MLP(hidden_size=args.teacher_hidden, dropout=args.teacher_dropout)
    t0 = time.time()
    teacher = train_hard_targets(
        teacher, train_loader_jitter, test_loader,
        epochs=args.epochs_teacher, lr=args.lr, device=device,
    )
    teacher_errors, teacher_acc = evaluate(teacher, test_loader, device)
    results["teacher"] = teacher_errors
    print(f"--> Professor final: {teacher_errors} erros "
          f"({teacher_acc*100:.2f}% acc)  [{time.time()-t0:.1f}s]\n")

    # ---- 2) Aluno baseline: rede pequena, SEM dropout, treinada
    #        normalmente com rótulos verdadeiros -----------------------
    print("=== Treinando o ALUNO baseline (rede pequena, sem regularização) ===")
    student_baseline = MLP(hidden_size=args.student_hidden, dropout=0.0)
    t0 = time.time()
    student_baseline = train_hard_targets(
        student_baseline, train_loader_plain, test_loader,
        epochs=args.epochs_student, lr=args.lr, device=device,
    )
    base_errors, base_acc = evaluate(student_baseline, test_loader, device)
    results["student_baseline"] = base_errors
    print(f"--> Aluno baseline final: {base_errors} erros "
          f"({base_acc*100:.2f}% acc)  [{time.time()-t0:.1f}s]\n")

    # ---- 3) Aluno destilado: mesma arquitetura pequena, treinada
    #        por destilação em várias temperaturas ----------------------
    print("=== Treinando o ALUNO por DESTILAÇÃO (mesma rede pequena) ===")
    for T in args.temperatures:
        print(f"\n-- Temperatura T={T} --")
        student_distilled = MLP(hidden_size=args.student_hidden, dropout=0.0)
        t0 = time.time()
        student_distilled = train_distillation(
            student_distilled, teacher, train_loader_plain, test_loader,
            epochs=args.epochs_student, lr=args.lr, T=T, alpha=args.alpha,
            device=device,
        )
        dist_errors, dist_acc = evaluate(student_distilled, test_loader, device)
        results[f"student_distilled_T{T}"] = dist_errors
        print(f"--> Aluno destilado (T={T}) final: {dist_errors} erros "
              f"({dist_acc*100:.2f}% acc)  [{time.time()-t0:.1f}s]")

    # ---- Resumo final, no estilo da Tabela do artigo -------------------
    print("\n" + "=" * 60)
    print("RESUMO (comparar com: professor=67, baseline=146, dist(T=20)=74)")
    print("=" * 60)
    for k, v in results.items():
        print(f"  {k:30s}: {v} erros de teste")

    # ---- Experimento extra: dígito '3' nunca visto ---------------------
    if args.run_missing_digit_experiment:
        best_T = args.temperatures[len(args.temperatures) // 2]
        experiment_missing_digit(
            teacher, T=best_T, alpha=args.alpha,
            epochs=args.epochs_student, lr=args.lr,
            device=device, data_dir=args.data_dir,
        )


if __name__ == "__main__":
    main()


Usando dispositivo: cpu



100%|██████████| 9.91M/9.91M [00:00<00:00, 19.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.42MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.00MB/s]


=== Treinando o PROFESSOR (rede grande, com dropout) ===
  [hard]  epoch  1/15  test_acc=96.53%  test_errors=347
  [hard]  epoch  2/15  test_acc=97.38%  test_errors=262
  [hard]  epoch  3/15  test_acc=97.84%  test_errors=216
  [hard]  epoch  4/15  test_acc=97.99%  test_errors=201
  [hard]  epoch  5/15  test_acc=98.19%  test_errors=181
  [hard]  epoch  6/15  test_acc=98.16%  test_errors=184
  [hard]  epoch  7/15  test_acc=98.33%  test_errors=167
  [hard]  epoch  8/15  test_acc=98.24%  test_errors=176
  [hard]  epoch  9/15  test_acc=98.12%  test_errors=188
  [hard]  epoch 10/15  test_acc=98.57%  test_errors=143
  [hard]  epoch 11/15  test_acc=98.60%  test_errors=140
  [hard]  epoch 12/15  test_acc=98.63%  test_errors=137
  [hard]  epoch 13/15  test_acc=98.73%  test_errors=127
  [hard]  epoch 14/15  test_acc=98.64%  test_errors=136
  [hard]  epoch 15/15  test_acc=98.61%  test_errors=139
--> Professor final: 139 erros (98.61% acc)  [614.0s]

=== Treinando o ALUNO baseline (rede pequena, se

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import time

# ==========================================
# 1. Hiperparâmetros e Configurações
# ==========================================
BATCH_SIZE = 128
EPOCHS = 5 # Mantido baixo para execução rápida. Aumente para melhores resultados.
LEARNING_RATE = 0.001
TEMPERATURE = 5.0 # Parâmetro de temperatura para suavizar as predições
ALPHA = 0.5 # Peso entre a perda de destilação e a perda padrão

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executando no dispositivo: {device}\n")

# Transformações e Carregamento do Dataset MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 2. Definição das Redes Neurais
# ==========================================
class TeacherNet(nn.Module):
    """Modelo grande e complexo"""
    def __init__(self):
        super(TeacherNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 1200)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1200, 1200)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(1200, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

class StudentNet(nn.Module):
    """Modelo pequeno e leve"""
    def __init__(self):
        super(StudentNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ==========================================
# 3. Funções de Treinamento e Avaliação
# ==========================================
def train_standard(model, loader, epochs, lr, title):
    print(f"--- Treinando {title} ---")
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        print(f"Época {epoch}/{epochs} concluída.")
    print("")

def train_distillation(teacher, student, loader, epochs, lr, T, alpha):
    print("--- Treinando Aluno com Destilação (Knowledge Distillation) ---")
    teacher.eval() # Professor não é treinado, apenas faz inferência
    student.train()
    optimizer = optim.Adam(student.parameters(), lr=lr)

    # KLDivLoss precisa que as entradas sejam log-probabilidades
    criterion_kd = nn.KLDivLoss(reduction='batchmean')
    criterion_ce = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()

            # Predições do Aluno
            student_logits = student(data)

            # Predições do Professor (sem calcular gradientes)
            with torch.no_grad():
                teacher_logits = teacher(data)

            # 1. Perda de Cross Entropy Padrão (Hard Targets)
            loss_ce = criterion_ce(student_logits, target)

            # 2. Perda de Destilação (Soft Targets) usando Kullback-Leibler
            # Aluno usa log_softmax, Professor usa softmax
            student_soft = F.log_softmax(student_logits / T, dim=1)
            teacher_soft = F.softmax(teacher_logits / T, dim=1)
            loss_kd = criterion_kd(student_soft, teacher_soft) * (T * T)

            # Combinação das perdas
            loss = (1. - alpha) * loss_ce + alpha * loss_kd

            loss.backward()
            optimizer.step()
        print(f"Época {epoch}/{epochs} concluída.")
    print("")

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    accuracy = 100 * correct / total
    return accuracy, all_targets, all_preds

# ==========================================
# 4. Execução do Experimento
# ==========================================
# Instanciar modelos
teacher_model = TeacherNet().to(device)
student_base = StudentNet().to(device)
student_distilled = StudentNet().to(device)

# Treinar Professor (Teacher)
train_standard(teacher_model, train_loader, EPOCHS, LEARNING_RATE, "Professor (Teacher)")

# Treinar Aluno Padrão (Sem Destilação)
train_standard(student_base, train_loader, EPOCHS, LEARNING_RATE, "Aluno Padrão (Baseline)")

# Treinar Aluno com Destilação
train_distillation(teacher_model, student_distilled, train_loader, EPOCHS, LEARNING_RATE, TEMPERATURE, ALPHA)

# ==========================================
# 5. Avaliação e Impressão dos Resultados
# ==========================================
acc_teacher, y_true_t, y_pred_t = evaluate(teacher_model, test_loader)
acc_student_base, y_true_s, y_pred_s = evaluate(student_base, test_loader)
acc_student_dist, y_true_sd, y_pred_sd = evaluate(student_distilled, test_loader)

# Tabela de Comparação
print("\n" + "="*50)
print(f"{'TABELA DE COMPARAÇÃO DE RESULTADOS':^50}")
print("="*50)
print(f"{'Modelo':<35} | {'Acurácia (%)':>12}")
print("-" * 50)
print(f"{'Professor (Grande/Pesado)':<35} | {acc_teacher:>11.2f}%")
print(f"{'Aluno Padrão (Sem Destilação)':<35} | {acc_student_base:>11.2f}%")
print(f"{'Aluno Distilado (Com Soft Targets)':<35} | {acc_student_dist:>11.2f}%")
print("="*50 + "\n")

# Matrizes de Confusão
print("MATRIZ DE CONFUSÃO: ALUNO PADRÃO (SEM DESTILAÇÃO)")
print("-" * 50)
cm_base = confusion_matrix(y_true_s, y_pred_s)
print(cm_base)
print("\n" + "-" * 50)

print("MATRIZ DE CONFUSÃO: ALUNO DISTILADO")
print("-" * 50)
cm_dist = confusion_matrix(y_true_sd, y_pred_sd)
print(cm_dist)
print("\n" + "-" * 50)

Executando no dispositivo: cpu



100%|██████████| 9.91M/9.91M [00:00<00:00, 130MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 30.5MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 98.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.60MB/s]


--- Treinando Professor (Teacher) ---
Época 1/5 concluída.
Época 2/5 concluída.
Época 3/5 concluída.
Época 4/5 concluída.
Época 5/5 concluída.

--- Treinando Aluno Padrão (Baseline) ---
Época 1/5 concluída.
Época 2/5 concluída.
Época 3/5 concluída.
Época 4/5 concluída.
Época 5/5 concluída.

--- Treinando Aluno com Destilação (Knowledge Distillation) ---
Época 1/5 concluída.
Época 2/5 concluída.
Época 3/5 concluída.
Época 4/5 concluída.
Época 5/5 concluída.


        TABELA DE COMPARAÇÃO DE RESULTADOS        
Modelo                              | Acurácia (%)
--------------------------------------------------
Professor (Grande/Pesado)           |       97.61%
Aluno Padrão (Sem Destilação)       |       96.97%
Aluno Distilado (Com Soft Targets)  |       96.44%

MATRIZ DE CONFUSÃO: ALUNO PADRÃO (SEM DESTILAÇÃO)
--------------------------------------------------
[[ 969    0    2    1    1    0    4    1    2    0]
 [   0 1123    5    0    0    1    3    1    2    0]
 [   3    2 1003    7  